# PDF Input Quickstart for Qwen3.5-27B

この Notebook は、英語論文 PDF を **text として入れる方法** と **画像化して vision input として入れる方法** をまとめたものです。サンプルとして arXiv の深層学習系論文 2 本を使います。


## 1. 前提
- Docker で SGLang API サーバが起動している
- `OPENAI_BASE_URL` を必要に応じて設定する
- PDF サンプルは `papers/` に配置済み


In [1]:
import os
from pathlib import Path
from openai import OpenAI

BASE_URL = os.environ.get('OPENAI_BASE_URL', 'http://127.0.0.1:30000/v1')
PAPERS = Path('../papers')
print('Using base_url =', BASE_URL)
print('Papers:', sorted(p.name for p in PAPERS.glob('*.pdf')))
client = OpenAI(api_key='EMPTY', base_url=BASE_URL)
client


Using base_url = http://127.0.0.1:30009/v1
Papers: ['attention_is_all_you_need.pdf', 'deep_residual_learning.pdf']


## 2. PDF から text だけ抽出して入れる


In [2]:
from pypdf import PdfReader

pdf_path = PAPERS / 'attention_is_all_you_need.pdf'
reader = PdfReader(str(pdf_path))
text = '\n'.join(page.extract_text() or '' for page in reader.pages[:3])
print(text[:3000])


Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Ex

In [3]:
resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[
        {'role': 'system', 'content': 'You are reading an English deep learning paper extracted from PDF text.'},
        {'role': 'user', 'content': 'Summarize the following paper excerpt in Japanese in 5 bullet points.\n\n' + text[:12000]},
    ],
    max_tokens=256,
)
resp


ChatCompletion(id='786dfbe080fb40a6900d3184db5dcbc4', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content='Thinking Process:\n\n1.  **Analyze the Request:**\n    *   **Task:** Summarize the provided paper excerpt in Japanese.\n    *   **Format:** 5 bullet points.\n    *   **Source:** An excerpt from the paper "Attention Is All You Need" (Vaswani et al., 2017).\n    *   **Content:** The excerpt includes the title, authors, abstract, introduction, background, and part of the model architecture section.\n\n2.  **Analyze the Source Text:**\n    *   **Title:** Attention Is All You Need.\n    *   **Authors:** Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Gomez, Kaiser, Polosukhin (Google Brain/Research, U of Toronto).\n    *   **Abstract:**\n        *   Current dominant models use RNNs or CNNs with attention.\n        *   P

## 3. PDF を画像化して vision input として入れる


In [4]:
import fitz  # pymupdf

pdf_path = PAPERS / 'deep_residual_learning.pdf'
doc = fitz.open(pdf_path)
page = doc.load_page(0)
pix = page.get_pixmap(matrix=fitz.Matrix(1.5, 1.5))
out_dir = Path('../papers/rendered')
out_dir.mkdir(parents=True, exist_ok=True)
img_path = out_dir / 'deep_residual_learning_page1.png'
pix.save(img_path)
img_path


PosixPath('../papers/rendered/deep_residual_learning_page1.png')

In [5]:
import base64
import mimetypes

mime = mimetypes.guess_type(img_path.name)[0] or 'image/png'
image_url = 'data:' + mime + ';base64,' + base64.b64encode(img_path.read_bytes()).decode('utf-8')

resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[
        {
            'role': 'user',
            'content': [
                {'type': 'text', 'text': 'This is the first page of an English deep learning paper PDF rendered as an image. Summarize what kind of paper it is in Japanese.'},
                {'type': 'image_url', 'image_url': {'url': image_url}},
            ],
        }
    ],
    max_tokens=256,
)
resp


ChatCompletion(id='425eb06708c84ff08c5cda58027c1c5b', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content='ユーザーは、提供された画像（英語の深層学習論文の1ページ目）を要約し、それがどのような論文か日本語で説明することを求めています。\n\n**画像の分析:**\n1.  **タイトル:** "Deep Residual Learning for Image Recognition"（画像認識のための深層残差学習）\n2.  **著者:** Kaiming He, Xiangyu Zhang, Shaoqing Ren, Jian Sun (Microsoft Research)\n3.  **Abstract (抄録):**\n    *   深いニューラルネットワークは訓練が難しいという問題提起。\n    *   「残差学習フレームワーク」を提案し、より深いネットワークの訓練を容易にすることを目指している。\n    *   層を参照なしの関数として学習するのではなく、層入力に対する残差関数として明示的に再定式化する。\n    *   残差ネットワークは最適化が容易で、深さを増やすことで精度を向上できるという証拠を示している。\n    *   ImageNetデータセットで最大152層の残差ネットを評価し、VGGネットよりも深いにもかかわらず低複雑度でありながら高い精度を達成。\n    *   ILSVRC'), matched_stop=None)], created=1774938943, model='Qwen/Qwen3.5-27B', object='chat.completion', service_tier=None, system_fingerprint=None, usage=Complet

## 4. 補足
- text 抽出は本文検索・要約に向く
- 画像化は図表・レイアウト・数式を含めて見せたい時に向く
